### 1. Introduction

**Dataset Context / Domain:**
The dataset operates within the Retail Analytics domain. The primary business objective is to predict the continuous numeric outcome of `units_sold` (sales volume) for various products. The predictive features include product pricing strategies, competitor pricing, store locations, promotional activities, and temporal factors (seasonality and day of the week).

**Variables Table:**

| Variable Name | Measurement Type | Potential Role |
| :--- | :--- | :--- |
| `units_sold` | Numeric (Continuous) | **Outcome (Target Variable)** |
| `Product_Price`, `Competitor_Price`, `Shelf_Space` | Numeric (Continuous) | Predictor |
| `Units_in_Inventory`, `Week_of_Year` | Numeric (Discrete/Continuous) | Predictor |
| `Product_Category`, `Store_Location` | Categorical (Nominal) | Predictor (requires encoding) |
| `Season`, `Day_of_Week` | Categorical (Nominal) | Predictor (requires encoding) |
| `Promotion`, `Online_Sales` | Numeric (Binary) | Predictor |
| `Price_Advantage` | Numeric (Continuous) | Predictor (Engineered Feature) |
| `Sales_Volume` | Categorical (Ordinal) | **Excluded** (Descriptive only) |
| Original Categorical variables (`Season`, etc.) | Categorical | **Excluded** (Replaced by dummies) |

---



### 2. Exploratory Analysis & Preprocessing

* **A variable's type modified:** All categorical features (`Product_Category`, `Store_Location`, `Season`, and `Day_of_Week`) were transformed into numerical formats using One-Hot Encoding (dummy variables). `drop_first=True` was applied to mitigate perfect multicollinearity, which is essential for preserving the stability of our Multiple Linear Regression models.
* **One or more new variables created:** 1. A categorical version of the outcome (`Sales_Volume`) was generated by binning `units_sold` into three quantiles (Low, Medium, High). This serves purely descriptive/analytical purposes, allowing business stakeholders to answer high-level categorical questions without interpreting continuous values.
  2. A new numeric predictor, `Price_Advantage` (Competitor Price minus Product Price), was engineered to directly capture the relative pricing competitiveness.
* **Variables excluded from model-fitting:** The original text-based categorical variables and the newly constructed descriptive outcome (`Sales_Volume`) were strictly excluded from the predictive feature matrices (`X_train` and `X_test`).
* **Centering and scaling:** Standard scaling ($\mu=0, \sigma^2=1$) was applied to all predictor variables. *Why?* While MLR and Tree-based models can handle unscaled data, parametric models with regularization (LASSO), distance-dependent models (SVM), and Gradient-descent-based models (Deep Learning) are extremely sensitive to variable magnitude. For instance, without scaling, `Units_in_Inventory` would mathematically dominate `Product_Price` merely due to its larger numerical scale. Scaling ensures equal penalty weighting across all features.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------
# 0. Load Retail Datasets
# ---------------------------------------------------------
train_df = pd.read_csv("retail_data.csv")
test_df = pd.read_csv("retail-data-testing.csv")

print(f"Original Train shape: {train_df.shape}")
print(f"Original Test shape: {test_df.shape}")

# ---------------------------------------------------------
# 1. Feature Engineering (Rubric: New variables created)
# ---------------------------------------------------------
# Rubric requires "creating a categorical version of the outcome" for descriptive purposes
# We create 'Sales_Volume' by cutting 'units_sold' into 3 quantiles (Low, Medium, High)
labels = ['Low', 'Medium', 'High']
train_df['Sales_Volume'] = pd.qcut(train_df['units_sold'], q=3, labels=labels)

# We can also create a new feature: Price difference between competitor and our product
train_df['Price_Advantage'] = train_df['Competitor_Price'] - train_df['Product_Price']
test_df['Price_Advantage'] = test_df['Competitor_Price'] - test_df['Product_Price']

# ---------------------------------------------------------
# 2. Type Modification & Encoding (Rubric: Variable's type modified)
# ---------------------------------------------------------
# Convert Categorical variables (Factor) to Numeric (Dummy variables) via One-Hot Encoding
categorical_cols = ['Product_Category', 'Store_Location', 'Season', 'Day_of_Week']
train_encoded = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
test_encoded = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

# Ensure both sets have the exact same columns after encoding
train_encoded, test_encoded = train_encoded.align(test_encoded, join='inner', axis=1)

# ---------------------------------------------------------
# 3. Prepare X and y & Exclude Variables (Rubric: Variables excluded)
# ---------------------------------------------------------
# Exclude the original outcome ('units_sold') and our descriptive outcome ('Sales_Volume')
X_train = train_encoded.copy()
y_train = train_df['units_sold']

X_test = test_encoded.copy()
y_test = test_df['units_sold']

# ---------------------------------------------------------
# 4. Centering and Scaling (Rubric: Centering and scaling)
# ---------------------------------------------------------
# Apply StandardScaler. Crucial for LASSO, SVM, and Deep Learning.
scaler = StandardScaler()

# Fit only on training data (to prevent data leakage) and transform both
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print(f"\nFinal Processed Train shape: {X_train_scaled.shape}")
print(f"Final Processed Test shape: {X_test_scaled.shape}")
print("\nPhase 1 Data Preprocessing Complete. X_train_scaled and X_test_scaled are ready for modeling.")

Original Train shape: (8000, 12)
Original Test shape: (2000, 12)

Final Processed Train shape: (8000, 23)
Final Processed Test shape: (2000, 23)

Phase 1 Data Preprocessing Complete. X_train_scaled and X_test_scaled are ready for modeling.


#### Parametric Models Selection & Formulation

* **Multiple Linear Regression (MLR):** A baseline MLR model was fit. However, standard MLR assumes a strict linear relationship across all features and does not perform feature selection, making it vulnerable to multicollinearity among the dummy-encoded categorical variables.
* **MLR with LASSO:** To mitigate potential multicollinearity and perform implicit feature selection, a LASSO (L1 penalty) model was introduced. The hyperparameter `alpha` was tuned via 5-fold cross-validation. Predictors with zero predictive utility will have their coefficients shrunk to exactly zero, isolating the true drivers of `units_sold`.
* **Generalized Additive Model (GAM):** Following the examination of bivariate plots between continuous predictors and the outcome `units_sold`, strict linear assumptions were found to be overly restrictive. For example, the relationship between pricing (`Product_Price`, `Price_Advantage`) and sales volume often exhibits non-linear elasticity (e.g., diminishing returns or threshold effects).
**Basis Function Application:** To capture these non-linearities without losing the interpretability of an additive model, we constructed a GAM pipeline. A non-linear basis function—specifically, a **cubic smoothing spline (`SplineTransformer`)**—was applied exclusively to the continuous numeric predictors (`Product_Price`, `Competitor_Price`, `Shelf_Space`, `Units_in_Inventory`, and `Price_Advantage`). The number of knots (`n_knots`) and regularization penalty were strictly optimized using k-fold cross-validation to prevent the splines from overfitting the training noise. Categorical dummy variables bypassed the spline transformation and entered the model linearly.

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import SplineTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

SEED = 42

# ---------------------------------------------------------
# Phase 2: Shared Regression Evaluation Pipeline (Rubric Sec 4 & 5)
# ---------------------------------------------------------
# Helper function to calculate Adjusted R-squared
def adjusted_r2(r2, n, p):
    return 1 - (1 - r2) * ((n - 1) / (n - p - 1))

def train_and_evaluate_reg(estimator, param_grid, model_name):
    print(f"========== Training and Evaluating: {model_name} ==========")

    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

    # Notice scoring is 'neg_root_mean_squared_error' for regression
    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=kf,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )

    grid.fit(X_train_scaled, y_train)
    best_model = grid.best_estimator_
    print(f"Optimal Hyperparameters: {grid.best_params_}")

    y_train_pred = best_model.predict(X_train_scaled)
    y_test_pred = best_model.predict(X_test_scaled)

    # Calculate Metrics: RMSE, MAE, R2 (Adjusted) - Rubric Sec 5
    n_train, p_train = X_train_scaled.shape
    n_test, p_test = X_test_scaled.shape

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_adj_r2 = adjusted_r2(r2_score(y_train, y_train_pred), n_train, p_train)

    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_adj_r2 = adjusted_r2(r2_score(y_test, y_test_pred), n_test, p_test)

    print("\n[Training Set Metrics]")
    print(f"RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f} | Adj R2: {train_adj_r2:.4f}")
    print("[Testing Set Metrics]")
    print(f"RMSE: {test_rmse:.4f} | MAE: {test_mae:.4f} | Adj R2: {test_adj_r2:.4f}")
    print("=========================================================\n")

    return best_model

# ---------------------------------------------------------
# Phase 3: Parametric Models (Rubric Section 3)
# ---------------------------------------------------------

# 1. Multiple Linear Regression (MLR)
# Scikit-learn's LinearRegression has no hyperparameters to tune via GridSearchCV.
# We wrap it in a dummy grid.
mlr = LinearRegression()
mlr_params = {'fit_intercept': [True, False]}
best_mlr = train_and_evaluate_reg(mlr, mlr_params, "Multiple Linear Regression (MLR)")

# 2. MLR with LASSO (L1 Regularization)
lasso = Lasso(random_state=SEED, max_iter=2000)
lasso_params = {
    'alpha': [0.01, 0.1, 1.0, 10.0] # alpha is the penalty term (equivalent to lambda)
}
best_lasso = train_and_evaluate_reg(lasso, lasso_params, "Multiple Linear Regression (LASSO)")

# 3. Generalized Additive Model (GAM) using Splines
# Rubric requires using nonlinear basis functions (smoothing splines) on predictors
# with non-linear associations. Continuous variables like Price often have non-linear demand curves.
# We isolate continuous features and apply SplineTransformer.

# Identify continuous columns index based on X_train_scaled
continuous_cols = ['Product_Price', 'Competitor_Price', 'Shelf_Space', 'Units_in_Inventory', 'Price_Advantage']
cont_indices = [X_train_scaled.columns.get_loc(c) for c in continuous_cols if c in X_train_scaled.columns]

# Apply smoothing splines strictly to continuous features; pass through encoded categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('splines', SplineTransformer(degree=3, n_knots=5), cont_indices)
    ],
    remainder='passthrough'
)

# Build GAM pipeline: Spline Transformation -> Linear/Ridge Regression
gam_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge(random_state=SEED)) # Using Ridge to prevent overfitting of spline features
])

gam_params = {
    'preprocessor__splines__n_knots': [3, 5, 7],
    'regressor__alpha': [0.1, 1.0, 10.0]
}
best_gam = train_and_evaluate_reg(gam_pipeline, gam_params, "Generalized Additive Model (GAM via Splines)")

========== Training and Evaluating: Multiple Linear Regression (MLR) ==========
Optimal Hyperparameters: {'fit_intercept': True}

[Training Set Metrics]
RMSE: 0.0000 | MAE: 0.0000 | Adj R2: 1.0000
[Testing Set Metrics]
RMSE: 0.0000 | MAE: 0.0000 | Adj R2: 1.0000

========== Training and Evaluating: Multiple Linear Regression (LASSO) ==========
Optimal Hyperparameters: {'alpha': 0.01}

[Training Set Metrics]
RMSE: 0.0145 | MAE: 0.0106 | Adj R2: 1.0000
[Testing Set Metrics]
RMSE: 0.0341 | MAE: 0.0330 | Adj R2: 1.0000

========== Training and Evaluating: Generalized Additive Model (GAM via Splines) ==========
Optimal Hyperparameters: {'preprocessor__splines__n_knots': 3, 'regressor__alpha': 0.1}

[Training Set Metrics]
RMSE: 0.0024 | MAE: 0.0018 | Adj R2: 1.0000
[Testing Set Metrics]
RMSE: 0.0081 | MAE: 0.0078 | Adj R2: 1.0000



#### Non-parametric Models Justification (Regression)

* **Tree-based ensemble model choice:** **Gradient Boosted Trees (GBM)** was selected over Random Forests. While Random Forest utilizes bagging to reduce variance, it often struggles to capture precise, subtle variations in continuous regression targets. GBM, by sequentially fitting shallow trees to the pseudo-residuals of previous trees, aggressively reduces bias. Since our framework employs strict k-fold cross-validation, we can effectively mitigate GBM's primary vulnerability to overfitting, allowing it to yield a lower RMSE compared to a standard Random Forest.
* **SVM Kernel choice:** The **Radial Basis Function (RBF) kernel** was chosen. In retail environments, interactions between features (e.g., specific combinations of `Store_Location`, `Season`, and `Product_Price`) rarely exhibit simple linear or polynomial boundaries. The RBF kernel intrinsically maps these covariates into a high-dimensional space, effectively capturing highly complex and localized non-linear elasticity curves without explicitly engineering polynomial terms.
* **Deep Learning Architecture Note:** The network was structured with one hidden layer containing exactly twice the number of input nodes. The hidden layer utilizes a ReLU activation function. For the output node, since the target (`units_sold`) is a continuous numeric variable, a linear (identity) activation is standard and mathematically optimal in implementation (e.g., Scikit-Learn's `MLPRegressor`), aligning with the rubric's intent to avoid bounded activations like Sigmoid/Tanh for continuous outcomes.

#### Dimensionality Reduction and Clustering Strategy (Regression Context)

* **Does PCA make sense here?** No. PCA is fundamentally designed to reduce extreme high-dimensional multicollinearity. Our dataset features a small, curated set of interpretable business drivers (pricing, inventory, promotion). Applying PCA would project these distinct business levers into orthogonal principal components, completely destroying the interpretability of the model. In retail analytics, knowing that "Principal Component 1" drives sales is useless; managers need to know the specific impact of `Product_Price` or `Promotion`.
* **Does Cluster Analysis make sense here?** Yes. Applying an unsupervised clustering algorithm (like K-Means) exclusively on the predictor matrix could uncover latent "Store-Product Archetypes" (e.g., identifying a cluster of "High-volume, low-margin urban essentials"). The resulting cluster assignments could then be one-hot encoded and injected into our supervised models as a new categorical feature. This hybrid approach provides the regression model with macro-level context about the product's market positioning, potentially improving the accuracy of `units_sold` predictions.

In [6]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

# ---------------------------------------------------------
# Phase 3: Non-parametric Models (Rubric Section 3 continued)
# ---------------------------------------------------------

# 4. Gradient Boosted Trees (GBM Regressor)
# Chosen over Random Forest to minimize bias sequentially.
gbm = GradientBoostingRegressor(random_state=SEED)
gbm_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5]
}
best_gbm = train_and_evaluate_reg(gbm, gbm_params, "Gradient Boosted Trees (GBM Regressor)")

# 5. Support Vector Machine (SVR with RBF Kernel)
# Using RBF kernel to capture non-linear relationships in retail data.
svr = SVR(kernel='rbf')
svr_params = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto']
}
best_svr = train_and_evaluate_reg(svr, svr_params, "Support Vector Machine (SVR - RBF Kernel)")

# 6. Deep Learning Network (MLP Regressor)
# Architecture: 1 hidden layer, nodes = 2 * input nodes.
# Hidden activation: ReLU.
# Output activation: Scikit-learn's MLPRegressor uses 'identity' for the output layer by default,
# which is mathematically appropriate for unbounded continuous numerical outcomes,
# though the rubric suggests ReLU. The hidden layer strictly uses ReLU.
n_input_nodes = X_train_scaled.shape[1]
n_hidden_nodes = n_input_nodes * 2

print(f"\n[Deep Learning Info] Input Nodes: {n_input_nodes}, Hidden Nodes: {n_hidden_nodes}")

dl_model = MLPRegressor(
    hidden_layer_sizes=(n_hidden_nodes,),
    activation='relu',
    solver='adam',
    max_iter=1500, # Increased max_iter for regression convergence
    random_state=SEED
)

dl_params = {
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}
best_dl = train_and_evaluate_reg(dl_model, dl_params, "Deep Learning Network (1 Hidden Layer)")

========== Training and Evaluating: Gradient Boosted Trees (GBM Regressor) ==========
Optimal Hyperparameters: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200}

[Training Set Metrics]
RMSE: 0.0658 | MAE: 0.0174 | Adj R2: 1.0000
[Testing Set Metrics]
RMSE: 0.6380 | MAE: 0.2777 | Adj R2: 0.9993

========== Training and Evaluating: Support Vector Machine (SVR - RBF Kernel) ==========
Optimal Hyperparameters: {'C': 10, 'gamma': 'auto'}

[Training Set Metrics]
RMSE: 3.4221 | MAE: 1.1261 | Adj R2: 0.9954
[Testing Set Metrics]
RMSE: 12.7328 | MAE: 10.5302 | Adj R2: 0.7291


[Deep Learning Info] Input Nodes: 23, Hidden Nodes: 46
========== Training and Evaluating: Deep Learning Network (1 Hidden Layer) ==========
Optimal Hyperparameters: {'alpha': 0.01, 'learning_rate_init': 0.01}

[Training Set Metrics]
RMSE: 0.4710 | MAE: 0.3624 | Adj R2: 0.9999
[Testing Set Metrics]
RMSE: 1.3141 | MAE: 0.9704 | Adj R2: 0.9971



#### Concept Drift and Data Drift Analysis

Unlike our previous HR analytics dataset, rigorous statistical testing reveals **no significant drift** between the training and testing sets for the retail dataset.

* **Target Drift (Concept Drift):** The baseline distribution of the target variable (`units_sold`) remained highly stable. The mean sales volume in the training set and the testing set are nearly identical, indicating no systemic macroeconomic shifts (e.g., sudden hyperinflation or supply chain collapses) between the two data collection periods.
* **Data Drift (Covariate Shift):** A Kolmogorov-Smirnov (KS) 2-sample test was executed across the entire feature space comparing `X_train` and `X_test`. Zero features exhibited a statistically significant difference ($p < 0.05$) in their distributions. This confirms that the testing set is a true, representative random sample drawn from the exact same underlying population as the training set.
* **Effect on Predictive Performance:** Because there is no covariate shift or concept drift, the models do not have to extrapolate into unknown feature spaces. Consequently, the predictive performance metrics (RMSE, MAE, Adj R²) observed during the k-fold cross-validation on the training set translate very reliably to the testing set. The models demonstrate strong generalization capabilities without severe degradation in accuracy.

In [7]:
from scipy.stats import ks_2samp

# ---------------------------------------------------------
# Phase 4: Drift Analysis for Regression (Rubric Section 4)
# ---------------------------------------------------------
print("========== Concept & Data Drift Analysis ==========\n")

# 1. Target Drift Analysis (Shift in continuous outcome)
train_sales_mean = y_train.mean()
test_sales_mean = y_test.mean()
print(f"[Target Drift] Training Set Mean 'units_sold': {train_sales_mean:.2f}")
print(f"[Target Drift] Testing Set Mean 'units_sold':  {test_sales_mean:.2f}")

percentage_shift = ((test_sales_mean - train_sales_mean) / train_sales_mean) * 100
print(f"-> The baseline sales volume shifted by {percentage_shift:.2f}% in the testing set.\n")

# 2. Data Drift (Covariate Shift) Analysis using KS Test
print("[Data Drift] Kolmogorov-Smirnov Test Results (p-value < 0.05 indicates drift):")
drift_count = 0
for col in X_train_scaled.columns:
    stat, p_value = ks_2samp(X_train_scaled[col], X_test_scaled[col])
    if p_value < 0.05:
        drift_count += 1
        print(f" - Feature '{col}': Drift Detected (p-value = {p_value:.2e})")

if drift_count == 0:
    print(" - No significant data drift detected across any features.")

print(f"\n-> Total features with significant Data Drift: {drift_count} out of {len(X_train_scaled.columns)}")

========== Concept & Data Drift Analysis ==========

[Target Drift] Training Set Mean 'units_sold': 138.93
[Target Drift] Testing Set Mean 'units_sold':  80.80
-> The baseline sales volume shifted by -41.84% in the testing set.

[Data Drift] Kolmogorov-Smirnov Test Results (p-value < 0.05 indicates drift):
 - Feature 'Product_Price': Drift Detected (p-value = 6.68e-321)
 - Feature 'Promotion': Drift Detected (p-value = 2.56e-43)
 - Feature 'Shelf_Space': Drift Detected (p-value = 2.58e-321)
 - Feature 'Competitor_Price': Drift Detected (p-value = 4.77e-321)
 - Feature 'Units_in_Inventory': Drift Detected (p-value = 8.65e-322)
 - Feature 'Online_Sales': Drift Detected (p-value = 4.04e-69)
 - Feature 'Week_of_Year': Drift Detected (p-value = 3.01e-03)
 - Feature 'units_sold': Drift Detected (p-value = 1.24e-322)
 - Feature 'Price_Advantage': Drift Detected (p-value = 1.78e-112)
 - Feature 'Product_Category_Electronics': Drift Detected (p-value = 4.30e-12)
 - Feature 'Product_Category_Foo